# GNR638 - Deep Learning MCQ Solver

**Competition:** GNR638 Project - Visual MCQ Answering  
**Deadline:** 2nd May 2026, 23:59 UTC  
**Author:** Afnan Abdul Gafoor (22b2505)

---

## Approach Overview

This notebook solves deep learning multiple-choice questions presented as PNG images using a **Vision-Language Model (VLM)** in a **zero-shot inference** setup - no task-specific training is performed.

### Why Zero-Shot VLM?
- The model (**Qwen3-VL-8B-Instruct**) is pretrained on vast corpora covering deep learning textbooks, papers, and technical content - it already possesses the domain knowledge required.
- Fine-tuning on synthetic DL MCQ data risks **catastrophic forgetting** and topic-specific overfitting, since the test set covers unknown subtopics.
- The images are clean LaTeX-rendered text (not photographs), so OCR + reasoning via a capable VLM is sufficient.

### Model
**[Qwen3-VL-8B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct)** by Alibaba Cloud / Qwen Team.
- 8B parameter dense vision-language model
- State-of-the-art on MMMU, MathVista, DocVQA benchmarks
- Excels at document VQA and technical reasoning
- Separate **Instruct** variant (used here) is ~4× faster than the Thinking variant

### Hardware Strategy
| Hardware | Precision | Attention | Max Pixels |
|---|---|---|---|
| T4 (16 GB VRAM) | 4-bit NF4 (BnB) | SDPA | 448 × 448 |
| L40s (48 GB VRAM) | bfloat16 | SDPA | 1120 × 1120 |

### Scoring Reminder
```
final_score = correct - 0.25 × incorrect - hallucinated
```
- Predictions must be **1, 2, 3, 4** (options A–D) or **5** (skip)
- Any other value → hallucination penalty of **−1**
- Strategic skipping when uncertain is safer than guessing

---

## References
- Qwen3-VL Technical Report: https://arxiv.org/abs/2511.21631
- Qwen3-VL GitHub: https://github.com/QwenLM/Qwen3-VL
- BitsAndBytes (4-bit quantization): https://github.com/TimDettmers/bitsandbytes
- HuggingFace Transformers: https://github.com/huggingface/transformers
- qwen-vl-utils: https://github.com/QwenLM/Qwen3-VL/tree/main/qwen-vl-utils

## Cell 1 - Install Dependencies

**Run this cell first with internet ON.**  
All packages are pinned to avoid version conflicts at eval time.

> `qwen-vl-utils` provides the `process_vision_info()` helper that handles image loading and patch computation for Qwen VL models.  
> `bitsandbytes` enables 4-bit NF4 quantization for T4 deployment.  
> `accelerate` is required by HuggingFace for `device_map="auto"`.

In [1]:
!pip install -q \
    "transformers>=4.52.0" \
    "accelerate>=0.34.0" \
    "bitsandbytes>=0.43.0" \
    "qwen-vl-utils>=0.0.8"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 55.3 MB/s eta 0:00:00


## Cell 2 - Imports & GPU Cleanup

Sets `PYTORCH_ALLOC_CONF=expandable_segments:True` to reduce CUDA memory fragmentation on T4.  
Also clears any leftover model from a previous session to avoid OOM on re-runs.

In [2]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import gc
import re
import time
import traceback
import warnings
import pandas as pd
from pathlib import Path

import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

warnings.filterwarnings("ignore")

# Clear any leftover model/processor from a previous Kaggle session
try:
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print("Cleared previous session.")
except NameError:
    pass

print(f"PyTorch  : {torch.__version__}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
GPU      : Tesla T4
VRAM     : 15.6 GB


## Cell 3 - Configuration

All tunable parameters in one place.

### Key design decisions

**`MAX_PIXELS` per hardware:**  
Controls how many visual patches the ViT processes. More patches = better OCR but more VRAM and slower inference.  
- T4 cap of `448×448` keeps peak VRAM under 14 GB (4-bit model ~5 GB + KV cache + image tokens)  
- L40s can use full `1120×1120` in BF16 comfortably within 48 GB

**`TEMPERATURE = 0.15` (T4) / `0.1` (L40s):**  
Low enough to be near-deterministic but avoids the repetition loops that `temperature=0` triggers in Qwen3 models.  
Per Qwen's official documentation, greedy decoding (`temp=0`) is explicitly discouraged for Qwen3 models.

**`MAX_NEW_TOKENS = 768` (T4) / `1024` (L40s):**  
768 is sufficient for full chain-of-thought on most DL MCQs while saving ~10 min on T4 vs 1024.  
`repetition_penalty=1.05` guards against the model looping on long outputs, which is the main cause of slow outlier images.

**`attn_implementation = "sdpa"`:**  
PyTorch's native Scaled Dot-Product Attention. Faster than `"eager"` on both T4 and L40s.  
Flash Attention 2 is not supported on T4 (Turing architecture) so SDPA is the best available option universally.

In [ ]:
# =============================================================================
# PATHS  - adjust this based on the structure of your dataset
# =============================================================================
BASE_DIR   = Path("/kaggle/input/datasets/afnanabdulgafoor/gnr638-project-test/test_dataset") # Modify this to your dataset path
IMAGES_DIR = BASE_DIR / "images"
TEST_CSV   = BASE_DIR / "test.csv"
OUTPUT_CSV = Path("submission.csv")

# =============================================================================
# MODEL
# =============================================================================
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
#
# WHY Instruct and NOT Thinking:
#   - Qwen3-VL Instruct and Thinking are separate model checkpoints (not switchable at runtime)
#   - Instruct is ~4x faster per image (4.5s vs 18.5s on HF demo benchmarks)
#   - With 50 images and a 1hr limit, Thinking would exceed the budget on T4
#   - Instruct still reasons step-by-step via chain-of-thought in its output

# =============================================================================
# HARDWARE-ADAPTIVE SETTINGS  (auto-detected below)
# =============================================================================

# Image resolution caps (pixels = width * height)
# These control the number of vision patches fed to the model - the primary OOM knob
MIN_PIXELS      = 128 * 128        # floor: don't go below this (images are already small/clean)
MAX_PIXELS_T4   = 448 * 448        # T4 safe ceiling   (~200K px → ~1000 patches)
MAX_PIXELS_L40S = 1120 * 1120      # L40s full quality (~1.25M px)

# Generation parameters
TEMPERATURE_T4   = 0.15   # slightly above greedy to avoid repetition loops (Qwen3 recommendation)
TEMPERATURE_L40S = 0.10   # BF16 is more numerically stable, can go slightly lower
MAX_NEW_TOKENS_T4   = 768   # saves ~10 min vs 1024 for 50 images, still enough for CoT
MAX_NEW_TOKENS_L40S = 1024  # full budget on L40s
REPETITION_PENALTY  = 1.05  # prevents output loops; Qwen docs suggest ~1.05 for Instruct mode

# Attention backend
# sdpa  = PyTorch Scaled Dot-Product Attention - universally supported, faster than eager
# flash_attention_2 = NOT supported on T4 (Turing arch); only on Ampere+ (A100, L40s, etc.)
ATTN_IMPL = "sdpa"

## Cell 4 - Hardware Detection & Precision Selection

Automatically selects the right precision and resolution based on available VRAM.

| VRAM | Precision | Why |
|---|---|---|
| < 40 GB (T4) | 4-bit NF4 via BitsAndBytes | Model weights ~5 GB; safe on 15 GB usable VRAM |
| ≥ 40 GB (L40s) | bfloat16 | Full precision; no quantization overhead |

**NF4 quantization config:**  
- `bnb_4bit_quant_type="nf4"` - NormalFloat4, best accuracy at 4-bit for normally-distributed weights  
- `bnb_4bit_use_double_quant=True` - quantizes the quantization constants too, saves ~0.4 GB extra  
- `bnb_4bit_compute_dtype=torch.float16` - dequantizes to FP16 for compute (T4 supports FP16 tensor cores)

In [4]:
def get_vram_gb() -> float:
    if torch.cuda.is_available():
        return torch.cuda.get_device_properties(0).total_memory / 1e9
    return 0.0

vram = get_vram_gb()
print(f"Detected VRAM: {vram:.1f} GB")

if vram >= 40:
    # L40s / A100 / H100 path - full BF16, high resolution
    LOAD_IN_4BIT   = False
    TORCH_DTYPE    = torch.bfloat16
    MAX_PIXELS     = MAX_PIXELS_L40S
    TEMPERATURE    = TEMPERATURE_L40S
    MAX_NEW_TOKENS = MAX_NEW_TOKENS_L40S
    bnb_config     = None
    print("Mode      : bfloat16 (L40s / large GPU)")
else:
    # T4 / Colab / Kaggle path - 4-bit NF4, reduced resolution
    LOAD_IN_4BIT   = True
    TORCH_DTYPE    = torch.float16
    MAX_PIXELS     = MAX_PIXELS_T4
    TEMPERATURE    = TEMPERATURE_T4
    MAX_NEW_TOKENS = MAX_NEW_TOKENS_T4
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",           # NormalFloat4 - best 4-bit quality for LLM weights
        bnb_4bit_use_double_quant=True,       # nested quantization saves ~0.4 GB extra
        bnb_4bit_compute_dtype=torch.float16, # FP16 compute; T4 has FP16 tensor cores
    )
    print("Mode      : 4-bit NF4 (T4 / small GPU)")

print(f"max_pixels: {MAX_PIXELS} ({int(MAX_PIXELS**0.5)}×{int(MAX_PIXELS**0.5)})")
print(f"temperature: {TEMPERATURE}")
print(f"max_new_tokens: {MAX_NEW_TOKENS}")
print(f"attn_impl : {ATTN_IMPL}")

Detected VRAM: 15.6 GB
Mode      : 4-bit NF4 (T4 / small GPU)
max_pixels: 200704 (448×448)
temperature: 0.15
max_new_tokens: 768
attn_impl : sdpa


## Cell 5 - Load Model & Processor

**Important notes:**
- `device_map="auto"` places model layers on available GPU(s) automatically
- On Kaggle T4x2, we intentionally avoid multi-GPU parallelism - single GPU is 5× faster due to overhead
- `trust_remote_code=True` is required for Qwen3-VL's custom model architecture
- The processor handles both text tokenization and image patch extraction

In [ ]:
print(f"Loading model: {MODEL_NAME}")
print("This will take ~2-3 min on T4 (downloading + quantizing), ~1 min on L40s ...\n")

t0 = time.time()

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=TORCH_DTYPE,
    device_map="auto",
    quantization_config=bnb_config,   # None on L40s (BF16), NF4 config on T4
    attn_implementation=ATTN_IMPL,    # sdpa: faster than eager, works on both T4 and L40s
    trust_remote_code=True,
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

load_time = time.time() - t0
print(f"\nModel loaded in {load_time:.1f}s")

if torch.cuda.is_available():
    used_gb  = torch.cuda.memory_allocated() / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM used after load: {used_gb:.1f} / {total_gb:.1f} GB ({100*used_gb/total_gb:.0f}%)")

Loading model: Qwen/Qwen3-VL-8B-Instruct
This will take ~2-3 min on T4 (downloading + quantizing), ~1 min on L40s ...



config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]


Model loaded in 115.9s
VRAM used after load: 1.6 / 15.6 GB (11%)


## Cell 6 - Prompts

### Prompt Design Rationale

**System prompt:**
- Establishes the model as a domain expert (activates relevant knowledge)
- Explicitly maps option letters to output digits (avoids ambiguity)
- Instructs "5" for uncertainty (exploits the skip-with-no-penalty rule)
- Demands a single digit as the final line (makes parsing reliable)

**User prompt:**
- Reinforces the step-by-step reasoning instruction
- Repeats the digit→letter mapping as a local reminder
- Kept brief to avoid inflating input token count

**Why step-by-step reasoning?**  
Chain-of-thought prompting consistently improves accuracy on technical MCQs by forcing the model to verify each step before committing to an answer. The `parse_answer()` function in Cell 7 strips the reasoning and extracts only the final digit.

In [6]:
SYSTEM_PROMPT = (
    "You are an expert in deep learning with thorough knowledge of "
    "neural network architectures, optimization, backpropagation, "
    "CNNs, RNNs, Transformers, and related topics.\n\n"
    "You will be shown an image containing a multiple-choice question (MCQ).\n\n"
    "Instructions:\n"
    "1. Read the question title, question body, and all four options carefully.\n"
    "2. Transcribe any formulas, code, or mathematical expressions exactly.\n"
    "3. Reason through the problem step by step - show your work.\n"
    "4. At the very end, output ONLY a single digit on its own line:\n"
    "      1  →  option A is correct\n"
    "      2  →  option B is correct\n"
    "      3  →  option C is correct\n"
    "      4  →  option D is correct\n"
    "      5  →  genuinely uncertain (safe skip - no penalty)\n\n"
    "IMPORTANT: Your absolute final line must be exactly one digit (1/2/3/4/5) "
    "and nothing else. Do not write anything after the digit."
)

USER_PROMPT = (
    "Examine this deep learning MCQ image carefully. "
    "Think step by step, then output your final answer as a single digit "
    "(1=A, 2=B, 3=C, 4=D, 5=skip if uncertain)."
)

print("Prompts set.")
print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")
print(f"User prompt length  : {len(USER_PROMPT)} chars")

Prompts set.
System prompt length: 844 chars
User prompt length  : 158 chars


## Cell 7 - Answer Parser

Robustly extracts the answer digit from raw model output.

### Parser Strategy (in priority order)
1. Strip any `<think>...</think>` block first (Qwen3 Instruct can emit partial thinking markup)
2. If output is already exactly `"1"–"5"` or `"A"–"D"`, return directly
3. Regex scan for explicit answer statements (`"the answer is 2"`, `"answer: B"`, etc.)
4. Regex scan for letter patterns in option-selection language
5. Default: return `5` (skip) - **never return an invalid value** (would cost −1)

The fail-safe default of `5` is critical: any output outside 1–5 is a hallucination costing −1 point.

In [ ]:
def parse_answer(text: str) -> int:
    """Extract the MCQ answer digit (1-4) or 5 (skip) from raw model output.
    Never returns a value outside 1-5. Defaults to 5 on any parse failure.
    """
    if not text or not text.strip():
        return 5

    # Step 1: Strip <think>...</think> block (Qwen3 may emit partial thinking markup)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    if not text:
        return 5

    # Step 2: Exact match - model output is already a bare digit or letter
    if text in {"1", "2", "3", "4", "5"}:
        return int(text)
    if text.upper() in {"A", "B", "C", "D"}:
        return {"A": 1, "B": 2, "C": 3, "D": 4}[text.upper()]

    letter_to_digit = {"A": 1, "B": 2, "C": 3, "D": 4}

    # Step 3: Numeric answer patterns (in order of specificity)
    num_patterns = [
        r"\bthe\s+(?:correct\s+)?answer\s+is\s+(?:option\s+)?([1-4])\b",
        r"\banswer[:\s]+([1-4])\b",
        r"\boption\s+([1-4])\b",
        r"\b([1-4])\s+(?:is\s+correct|is\s+the\s+(?:correct\s+)?answer)\b",
        r"\bchoose\s+([1-4])\b",
        r"\bselect\s+(?:option\s+)?([1-4])\b",
        r"\*\*([1-4])\*\*",                   # bold markdown
        r"(?:^|\n)\s*([1-4])\s*$",            # digit alone on a line
    ]
    for pat in num_patterns:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
        if m:
            return int(m.group(1))

    # Step 4: Letter answer patterns
    letter_patterns = [
        r"\bthe\s+(?:correct\s+)?answer\s+is\s+(?:option\s+)?([A-D])\b",
        r"\banswer[:\s]+([A-D])\b",
        r"\boption\s+([A-D])\b",
        r"\b([A-D])\s+(?:is\s+correct|is\s+the\s+(?:correct\s+)?answer)\b",
        r"(?:^|\n)\s*([A-D])\s*[.:\)]\s",     # "A. ", "A: ", "A) "
        r"\bchoice\s+([A-D])\b",
        r"\bselect\s+(?:option\s+)?([A-D])\b",
        r"\*\*([A-D])\*\*",                   # bold markdown
        r"(?:^|\n)\s*([A-D])\s*$",            # letter alone on a line
    ]
    for pat in letter_patterns:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
        if m:
            return letter_to_digit.get(m.group(1).upper(), 5)

    # Step 5: Total parse failure - safe skip (0 points, no penalty)
    return 5


# Quick sanity check
assert parse_answer("") == 5
assert parse_answer("The answer is 2") == 2
assert parse_answer("option B is correct") == 2
assert parse_answer("<think>long reasoning</think>\n3") == 3
assert parse_answer("definitely option A") == 1
assert parse_answer("some garbage xyz") == 5
assert parse_answer("**4**") == 4
print("Parser sanity checks passed.")

Parser sanity checks passed.


## Cell 8 - Single-Image Inference Function

### Key implementation details

**`min_pixels` and `max_pixels`:**  
Passed directly to `qwen-vl-utils` which resizes the image to fit within this pixel budget before patch extraction. Setting `min_pixels=128*128` prevents the model from getting near-blank patches on small images.

**`repetition_penalty=1.05`:**  
Divides logits of recently-generated tokens by this factor, discouraging loops. Qwen's official docs recommend ~1.05 for Instruct mode.

**`do_sample=(TEMPERATURE > 0)`:**  
Disables sampling if temperature is exactly 0, enabling greedy decoding. With temperature > 0, sampling is used (which Qwen3 strongly recommends over greedy to avoid repetition).

**OOM fallback:**  
Progressively halves `max_pixels` on OOM. Tried at 1/2 resolution then 1/4. If all fail, returns 5 (skip) rather than crashing the whole run.

In [8]:
def solve_mcq(img_path: Path, max_pixels: int = MAX_PIXELS) -> tuple[int, str]:
    """Run Qwen3-VL on a single MCQ image and return (answer_digit, raw_output).

    Parameters
    ----------
    img_path   : path to the PNG image
    max_pixels : upper pixel budget for image patches (reduce to avoid OOM)

    Returns
    -------
    (answer, raw) where answer is in {1,2,3,4,5} and raw is the full model output
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": str(img_path),
                    "min_pixels": MIN_PIXELS,   # floor: don't under-sample clean LaTeX images
                    "max_pixels": max_pixels,   # ceiling: key OOM knob
                },
                {"type": "text", "text": USER_PROMPT},
            ],
        },
    ]

    # Build text input from chat template
    text_input = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Extract and preprocess image patches
    image_inputs, video_inputs = process_vision_info(messages)

    # Tokenize text + encode image patches into model inputs
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    # Generate answer
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=(TEMPERATURE > 0),       # sampling if temp > 0 (Qwen3 recommendation)
            repetition_penalty=REPETITION_PENALTY,  # prevents looping on long outputs
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (not the input prompt)
    new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    raw = processor.batch_decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0]

    return parse_answer(raw), raw


print("solve_mcq() defined.")

solve_mcq() defined.


## Cell 9 - Main Inference Loop

Iterates over `test.csv`, runs `solve_mcq()` on each image, and collects results.

### Error handling
- **Image not found:** logs and skips (returns 5) - does not crash the loop
- **OOM:** retries at half resolution, then quarter resolution before giving up
- **Any other exception:** logs traceback, returns 5 (skip) - run completes regardless

### Memory management
`gc.collect()` + `torch.cuda.empty_cache()` between images frees fragmented VRAM. Critical on T4 where headroom is tight after model load.

In [9]:
test_df = pd.read_csv(TEST_CSV)
total   = len(test_df)
print(f"Test set: {total} images")
print(test_df.head(), "\n")

results   = []
t_run_start = time.time()

for idx, row in test_df.iterrows():
    image_name = row["image_name"]
    img_num    = idx + 1

    # Try the canonical image path
    img_path = IMAGES_DIR / f"{image_name}.png"
    if not img_path.exists():
        img_path = BASE_DIR / f"{image_name}.png"  # fallback: image at base level

    print(f"[{img_num:2d}/{total}] {image_name}", end="  ", flush=True)
    t_img = time.time()

    #  Case 1: Image file not found 
    if not img_path.exists():
        print("NOT FOUND → skip (5)")
        results.append({"image_name": image_name, "option": 5, "_raw": "image_not_found"})
        continue

    #  Case 2: Normal inference 
    try:
        option, raw = solve_mcq(img_path)
        elapsed = time.time() - t_img
        print(f"option={option}  ({elapsed:.1f}s)")
        print(f"    ↳ {raw[:140].strip()!r}")

    #  Case 3: OOM - retry at reduced resolution 
    except torch.cuda.OutOfMemoryError:
        print("OOM - retrying at reduced resolution ...")
        success = False

        for scale in [2, 4]:  # try ½ then ¼ of original pixel cap
            gc.collect()
            torch.cuda.empty_cache()
            reduced = MAX_PIXELS // scale
            print(f"    retrying at {int(reduced**0.5)}×{int(reduced**0.5)} (1/{scale} pixels) ...", end="  ")
            try:
                option, raw = solve_mcq(img_path, max_pixels=reduced)
                elapsed = time.time() - t_img
                print(f"option={option}  ({elapsed:.1f}s) [reduced res 1/{scale}]")
                success = True
                break
            except torch.cuda.OutOfMemoryError:
                print(f"OOM again at 1/{scale}")
            except Exception:
                print(f"ERROR at 1/{scale}:")
                traceback.print_exc()
                option, raw = 5, f"error_oom_retry_1/{scale}"
                success = True
                break

        if not success:
            print("All OOM retries exhausted → skip (5)")
            option, raw = 5, "error_oom_final"

    #  Case 4: Unexpected error 
    except Exception:
        print("ERROR:")
        traceback.print_exc()
        option, raw = 5, "error_unexpected"

    results.append({"image_name": image_name, "option": option, "_raw": raw})

    # Free VRAM between images - important on T4 where headroom is tight
    gc.collect()
    torch.cuda.empty_cache()

total_elapsed = time.time() - t_run_start
print(f"\nInference complete in {total_elapsed/60:.1f} min  ({total_elapsed/total:.1f}s/image avg)")

Test set: 50 images
  image_id image_name
0  image_1    image_1
1  image_2    image_2
2  image_3    image_3
3  image_4    image_4
4  image_5    image_5 

[ 1/50] image_1  option=2  (39.6s)
    ↳ 'I need to calculate H_out using the given formula for dilated convolution.\n\nThe formula provided is:\nH_out = ⌊(H_in + 2p - d(k-1) - 1) / s⌋'
[ 2/50] image_2  option=2  (63.2s)
    ↳ 'Let me think through this step by step.\n\nThe question is about He initialization (also called Kaiming initialization) and why it uses factor'
[ 3/50] image_3  option=1  (98.5s)
    ↳ 'Let me think through this step by step.\n\nThe question asks for the gradient of loss L w.r.t. gamma in Batch Normalization.\n\nRecall the batch'
[ 4/50] image_4  option=1  (98.1s)
    ↳ 'Let me solve this step by step.\n\nWe need to find the parameter ratio between depthwise separable convolutions and standard convolutions.\n\nGi'
[ 5/50] image_5  option=1  (68.9s)
    ↳ 'Let me analyze this PyTorch code for Convolutional Neura

## Cell 10 - Build & Validate Submission File

Validates the submission against the rules before saving:
- All options must be in {1, 2, 3, 4, 5} - anything else is a hallucination penalty (−1)
- Row count must match `test.csv` exactly
- Column names must match `sample_submission.csv` format: `image_name, option`

> **Note on format:** The `sample_submission.csv` has columns `image_name, option` (no `id` column).  
> The README prose mentions an `id` column but the actual sample file does not - we follow the file.

In [10]:
submission = pd.DataFrame(results)[["image_name", "option"]]

#  Validation 
assert list(submission.columns) == ["image_name", "option"], \
    f"Wrong columns: {list(submission.columns)}"

assert len(submission) == len(test_df), \
    f"Row count mismatch: {len(submission)} vs {len(test_df)}"

assert submission["option"].between(1, 5).all(), \
    f"Invalid option values: {submission[submission['option'].between(1,5) == False]}"

#  Save 
submission.to_csv(OUTPUT_CSV, index=False)

#  Summary 
dist    = submission["option"].value_counts().sort_index()
skipped = int((submission["option"] == 5).sum())
answered = total - skipped

print(f"{'='*55}")
print(f"Saved: {OUTPUT_CSV}  ({len(submission)} rows)")
print(f"{'='*55}")
print(submission.to_string(index=False))
print(f"\nDistribution : { {k: int(v) for k, v in dist.items()} }")
print(f"Answered     : {answered}/{total}")
print(f"Skipped      : {skipped}/{total}")
print(f"\nDone - submit submission.csv")

Saved: submission.csv  (50 rows)
image_name  option
   image_1       2
   image_2       2
   image_3       1
   image_4       1
   image_5       1
   image_6       5
   image_7       1
   image_8       3
   image_9       1
  image_10       3
  image_11       1
  image_12       2
  image_13       1
  image_14       1
  image_15       1
  image_16       1
  image_17       2
  image_18       2
  image_19       1
  image_20       2
  image_21       1
  image_22       2
  image_23       1
  image_24       2
  image_25       3
  image_26       2
  image_27       2
  image_28       2
  image_29       1
  image_30       4
  image_31       1
  image_32       2
  image_33       1
  image_34       1
  image_35       2
  image_36       1
  image_37       1
  image_38       2
  image_39       5
  image_40       2
  image_41       2
  image_42       1
  image_43       3
  image_44       2
  image_45       2
  image_46       1
  image_47       1
  image_48       2
  image_49       2
  image_50       